In [35]:
import torch
import torch.nn as nn

dtype = torch.float16

# ==========================================
# 1. Hyperparameters
# ==========================================
batch_size = 2
dim = 256
n_heads = 4
head_dim = dim // n_heads
vocab_size = 4069
seq_len = 120
max_seq_len = 128

# ==========================================
# 2. Parameters & Buffers
# ==========================================
# Weights
w_emb = nn.Parameter(torch.empty((vocab_size, dim), dtype=dtype))
wq = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
wk = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
wv = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
wo = nn.Parameter(torch.empty((dim, dim), dtype=dtype))
w_unemb = nn.Parameter(torch.empty((dim, vocab_size), dtype=dtype))

# RMSNorm Parameters
norm_attn = nn.Parameter(torch.ones(dim, dtype=dtype))
norm_final = nn.Parameter(torch.ones(dim, dtype=dtype))

# Initialize standard normal with std=0.02 (LLaMA standard)
for param in [w_emb, wq, wk, wv, wo, w_unemb]:
    nn.init.normal_(param, mean=0.0, std=0.02)

# KV Cache Buffers
cache_k = torch.empty((batch_size, n_heads, max_seq_len, head_dim), dtype=dtype)
cache_v = torch.empty((batch_size, n_heads, max_seq_len, head_dim), dtype=dtype)

def rms_norm(x, weight, eps=1e-5):
    # Upcast to float32 for variance to prevent float16 overflow
    variance = x.to(torch.float32).pow(2).mean(dim=-1, keepdim=True)
    return (x * torch.rsqrt(variance + eps).to(dtype)) * weight

# ==========================================
# 3. PREFILL STAGE (Full Context Processing)
# ==========================================
tokens = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long)
x = w_emb[tokens]

# Pre-Norm
x_norm = rms_norm(x, norm_attn)

# Q, K, V Projections
q = torch.matmul(x_norm, wq).reshape(batch_size, seq_len, n_heads, head_dim).transpose(1, 2)
k = torch.matmul(x_norm, wk).reshape(batch_size, seq_len, n_heads, head_dim).transpose(1, 2)
v = torch.matmul(x_norm, wv).reshape(batch_size, seq_len, n_heads, head_dim).transpose(1, 2)

# Write to Cache
cache_k[:, :, :seq_len, :] = k
cache_v[:, :, :seq_len, :] = v

# Attention & Causal Masking
attn = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
mask = torch.triu(torch.full((seq_len, seq_len), float('-inf'), dtype=dtype), diagonal=1)
attn = torch.softmax(attn + mask, dim=-1)

# Output Projection & Residual Connection
out = torch.matmul(attn, v).transpose(1, 2).reshape(batch_size, seq_len, dim)
x = x + torch.matmul(out, wo)

# Final Norm, Unembedding & Next Token Selection
x_norm = rms_norm(x, norm_final)
logits = torch.matmul(x_norm, w_unemb)
next_token = torch.argmax(logits[:, -1, :], dim=-1)

print("--- Prefill Completed ---")
print("Cache K shape:", cache_k.shape)
print("First Predicted Token shape:", next_token.shape)

# ==========================================
# 4. GENERATION STAGE (In-Place Cache Updates)
# ==========================================
pos = seq_len
n_steps = max_seq_len - seq_len

print("\n--- Generation Loop Started ---")
for step in range(n_steps):
    # Single Token Lookup
    x = w_emb[next_token].reshape(batch_size, 1, dim)
    x_norm = rms_norm(x, norm_attn)

    # Single Token Q, K, V Projections
    q = torch.matmul(x_norm, wq).reshape(batch_size, 1, n_heads, head_dim).transpose(1, 2)
    k = torch.matmul(x_norm, wk).reshape(batch_size, 1, n_heads, head_dim).transpose(1, 2)
    v = torch.matmul(x_norm, wv).reshape(batch_size, 1, n_heads, head_dim).transpose(1, 2)

    # Append to Cache
    cache_k[:, :, pos:pos + 1, :] = k
    cache_v[:, :, pos:pos + 1, :] = v
    pos += 1

    # Slice Past Cache for Attention
    k_past = cache_k[:, :, :pos, :]
    v_past = cache_v[:, :, :pos, :]

    # Cached Attention (No mask needed for single token generation)
    attn = torch.matmul(q, k_past.transpose(-2, -1)) / (head_dim ** 0.5)
    attn = torch.softmax(attn, dim=-1)

    # Output Projection & Residual Connection
    out = torch.matmul(attn, v_past).transpose(1, 2).reshape(batch_size, 1, dim)
    x = x + torch.matmul(out, wo)

    # Final Norm, Unembedding & Next Token Selection
    x_norm = rms_norm(x, norm_final)
    logits = torch.matmul(x_norm, w_unemb)
    next_token = torch.argmax(logits, dim=-1)

    print(f"Step {step + 1:02d} | Next Token: {next_token.flatten().tolist()}")

--- Prefill Completed ---
Cache K shape: torch.Size([2, 4, 128, 64])
First Predicted Token shape: torch.Size([2])

--- Generation Loop Started ---
Step 01 | Next Token: [1047, 3265]
Step 02 | Next Token: [1426, 3105]
Step 03 | Next Token: [957, 538]
Step 04 | Next Token: [1565, 1108]
Step 05 | Next Token: [1730, 529]
Step 06 | Next Token: [3139, 520]
Step 07 | Next Token: [2229, 698]
Step 08 | Next Token: [2338, 19]
